# 05 · Final Evaluation & Model Comparison

Loads best checkpoints for all models, runs test-set evaluation,
and generates all IEEE report figures.

**Checklist:**
- [ ] Notebooks 03 & 04 complete — checkpoints on Drive
- [ ] Runtime → **A100 GPU**

In [ ]:
import os, sys, json, glob, torch

GITHUB_USER = 'musarashid49'
REPO_NAME   = 'Image-Classification-with-CNN'
REPO_DIR    = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
LOCAL_DATA_DIR   = '/content/data'
DRIVE_SPLIT_PATH = '/content/drive/MyDrive/pk_politicians_split'
DRIVE_RESULTS    = '/content/drive/MyDrive/pk_politicians_results'

# Restore checkpoints from Drive
import shutil
if os.path.isdir(DRIVE_RESULTS):
    for f in os.listdir(os.path.join(DRIVE_RESULTS, 'checkpoints')):
        src = os.path.join(DRIVE_RESULTS, 'checkpoints', f)
        dst = os.path.join('results', 'checkpoints', f)
        os.makedirs('results/checkpoints', exist_ok=True)
        shutil.copy2(src, dst)
    print('✓ Checkpoints restored from Drive')
else:
    print('⚠  No Drive results folder found — run Notebooks 03 & 04 first')

In [ ]:
from src.utils import copy_dataset_from_drive, set_seed
from src.dataset import get_dataloaders

set_seed(42)
copy_dataset_from_drive(DRIVE_SPLIT_PATH, LOCAL_DATA_DIR)

loaders = get_dataloaders(
    os.path.join(LOCAL_DATA_DIR, 'train'),
    os.path.join(LOCAL_DATA_DIR, 'val'),
    os.path.join(LOCAL_DATA_DIR, 'test'),
)

In [ ]:
# Load checkpoints & evaluate all models
from src.models   import build_model
from src.utils    import load_checkpoint
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_misclassified

# Add/remove models from this dict as needed
CHECKPOINTS = {
    'resnet50':        'results/checkpoints/resnet50_best.pth',
    'efficientnet_b0': 'results/checkpoints/efficientnet_b0_best.pth',
    # 'efficientnet_b2': 'results/checkpoints/efficientnet_b2_best.pth',
}

all_results = {}
for model_name, ckpt_path in CHECKPOINTS.items():
    if not os.path.isfile(ckpt_path):
        print(f'⚠  Checkpoint not found: {ckpt_path}  (skipping)')
        continue
    print(f'\n══ {model_name} ═══════════════════════════')
    model = build_model(model_name, num_classes=16, dropout=0.4)
    load_checkpoint(model, ckpt_path, device)
    model.to(device)
    res = evaluate_model(model, loaders['test'], device, model_name=model_name)
    all_results[model_name] = res
    plot_confusion_matrix(res['y_true'], res['y_pred'], model_name=model_name).show()
    fig = plot_misclassified(res['images'], res['y_true'], res['y_pred'], model_name=model_name)
    if fig: fig.show()

In [ ]:
# Training curves from saved JSON history files
from src.evaluate import plot_training_curves

for hist_file in sorted(glob.glob('results/metrics/*_history.json')):
    name = os.path.basename(hist_file).replace('_history.json', '')
    with open(hist_file) as f:
        history = json.load(f)
    plot_training_curves(history, model_name=name).show()

In [ ]:
# Model comparison bar chart
from src.evaluate import plot_model_comparison

comparison_data = {
    name: {
        'accuracy': res['accuracy'],
        'macro_f1': res['report']['macro avg']['f1-score'],
    }
    for name, res in all_results.items()
}
plot_model_comparison(comparison_data).show()

In [ ]:
# Summary table
from scripts.export_results import build_summary_table, build_per_class_table

print('\n── Summary ──────────────────────────────────────────────────')
df = build_summary_table()
print(df.to_string(index=False))

print('\n── Per-class F1 pivot ───────────────────────────────────────')
df_cls = build_per_class_table()
if not df_cls.empty:
    pivot = df_cls.pivot(index='class', columns='model', values='f1')
    print(pivot.to_string())

In [ ]:
# Push all final results to Drive
from src.utils import save_results_to_drive
save_results_to_drive('results', DRIVE_RESULTS)

# Also copy plots to report/figures/
import shutil
os.makedirs('report/figures', exist_ok=True)
for png in glob.glob('results/plots/*.png'):
    shutil.copy2(png, os.path.join('report/figures', os.path.basename(png)))

print('\n✓ All results saved.')
print('✓ Plots copied to report/figures/ — use for IEEE Overleaf report.')